# Problem 7: First-Issue Finder

**Objective:** build a tool that finds beginner-friendly issues in any GitHub project and explains them in plain language, in support of Hacktoberfest.

Open-source projects track pending work on GitHub as **issues**. Many projects label tasks suitable for newcomers as **"good first issue"**, but these are often written in technical language. The tool will:

1. Use the **GitHub API** to fetch open beginner-friendly issues from any repository (for example, `ollama/ollama`)
2. Use an **open-weight AI model** to explain each issue: what it involves, its difficulty, the skills required, and a suggested first step
3. Show the results as structured cards with links to the original issues

**Learning outcomes:** calling a web API, combining data from two services, and obtaining structured JSON output from an AI model.

### How to use this notebook
- A notebook is made up of **cells**. To run a cell, select it and press **Shift + Enter** (or select the run button).
- On the first run, run the cells **in order, from top to bottom**.
- Comments marked **STUDENT TODO** show where changes are expected:
  - **Level 1:** text changes. No coding experience required.
  - **Level 2:** changes to settings and values.
  - **Level 3:** new features. Mentors are available to help.
- After changing a cell, **re-run that cell and then re-run the "Launch the application" cell** for the change to take effect.
- If the notebook stops working as expected, select **Runtime → Restart session** and run all cells again from the top.

### Obtaining a free Groq API key (one per team; about 2 minutes)

Groq is a free service that hosts **open-weight AI models**, such as OpenAI's gpt-oss. The code in this notebook sends a message to Groq and receives the model's response.

1. Go to **https://console.groq.com** and sign in with a Google or GitHub account.
2. Select **API Keys** → **Create API Key**, enter any name, and **copy** the key (it begins with `gsk_`).
3. In Colab, select the **key icon** in the left sidebar (**Secrets**).
4. Select **Add new secret**. Set **Name** to `GROQ_API_KEY` and **Value** to your key. Enable **Notebook access**.

> **Important:** do not paste the key directly into a code cell. Anyone the notebook is shared with would be able to see it. The Secrets panel keeps it private.
>
> **No key yet:** the notebook runs in **demonstration mode** with sample responses, so you can explore and edit it before a key is available.

### Optional: GitHub access token (only if a rate limit error is shown)
Without a token, GitHub allows about 60 requests per hour, shared by all users on the same network. To increase this limit: on github.com, go to Settings → Developer settings → Personal access tokens → **Fine-grained tokens** → Generate new token (no additional permissions are needed for public repositories). Add the token in Secrets under the name `GITHUB_TOKEN`.

## Step 0: Installation and setup (run both cells)

In [ ]:
# Installs the required libraries. This takes about 30 seconds.
!pip install -q "gradio>=5,<6" openai requests

In [ ]:
# SETUP: run this cell once. No changes are required here.
import os
import openai
from openai import OpenAI

def _get_secret(name):
    """Reads a secret from the Colab Secrets panel (or from an environment variable)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

GROQ_API_KEY = _get_secret("GROQ_API_KEY")
DEMO_MODE = not GROQ_API_KEY

# Groq uses the same interface as the OpenAI API, so the openai library is used
# and pointed at Groq's servers.
client = None if DEMO_MODE else OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")

def _friendly_error(e):
    if isinstance(e, openai.AuthenticationError):
        return "Error: the GROQ_API_KEY was rejected. Check that the complete key (beginning with gsk_) is saved in Colab Secrets, then re-run the setup cell."
    if isinstance(e, openai.RateLimitError):
        return "Error: the Groq free-tier usage limit has been reached. Wait about one minute and try again, or set MODEL = 'openai/gpt-oss-20b'."
    if isinstance(e, openai.NotFoundError):
        return f"Error: Groq does not recognise the model '{MODEL}'. Check the spelling; the model may also have been retired. See https://console.groq.com/docs/models"
    if isinstance(e, openai.APIConnectionError):
        return "Error: unable to connect to Groq. Check the internet connection and try again."
    return f"Error: the request to the AI service failed ({e})."

def ask_llm(messages, temperature=0.7, json_mode=False, demo_reply="(Demonstration mode response)"):
    """Sends a list of chat messages to the AI model and returns its response as text.
    In demonstration mode, returns `demo_reply` instead. If the request fails,
    returns a message beginning with "Error:" instead of stopping the program."""
    if DEMO_MODE:
        return demo_reply
    try:
        extra = {"response_format": {"type": "json_object"}} if json_mode else {}
        if MODEL.startswith("openai/gpt-oss"):
            extra["reasoning_effort"] = "low"  # gpt-oss models reason before answering; "low" keeps responses fast
        response = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=temperature, **extra
        )
        return response.choices[0].message.content
    except Exception as e:
        return _friendly_error(e)

if DEMO_MODE:
    print("Demonstration mode: no GROQ_API_KEY was found, so the application will show sample responses.")
    print("To connect the AI model: Secrets (left sidebar) → add GROQ_API_KEY → enable Notebook access → re-run this cell.")
else:
    print("Connected to Groq. The AI model is ready.")

GITHUB_TOKEN = _get_secret("GITHUB_TOKEN")
print("GitHub token found (higher rate limit)." if GITHUB_TOKEN else "No GitHub token found. This is sufficient for light use.")

## Step 1: Settings

In [ ]:
# STUDENT TODO (Level 1): set the application name, description and default repository.
APP_TITLE = "First-Issue Finder"
APP_DESCRIPTION = "Enter a GitHub repository to find beginner-friendly issues, explained in plain language."
DEFAULT_REPO = "ollama/ollama"

# STUDENT TODO (Level 2): the number of issues explained at a time (more issues are slower and use more of the free limit).
MAX_ISSUES = 5

# STUDENT TODO (Level 2): words in label names that indicate beginner-friendly issues (matching ignores case).
BEGINNER_LABEL_WORDS = ["good first", "beginner", "first-timer", "first timer", "easy", "starter", "newcomer"]

# STUDENT TODO (Level 2): the instructions for the AI. The tone or content of the explanations may be changed,
#   but keep the JSON format section so the application can read the response.
EXPLAIN_INSTRUCTIONS = """
You help first-year college students make their first open-source contribution.
For each GitHub issue provided, explain it in plain language, without jargon.
Reply with JSON only, in exactly this format:
{
  "issues": [
    {
      "number": 123,
      "summary": "1-2 plain-language sentences: what is the problem or task?",
      "difficulty": "Easy" or "Medium" or "Hard",
      "skills": ["skill 1", "skill 2"],
      "first_step": "one concrete first action a beginner should take"
    }
  ]
}
"""

TEMPERATURE = 0.3

# STUDENT TODO (Level 2): select the open-weight model that generates responses.
#   "openai/gpt-oss-20b"   OpenAI gpt-oss, 20 billion parameters. Fast and suitable for most tasks. (default)
#   "openai/gpt-oss-120b"  OpenAI gpt-oss, 120 billion parameters. More capable, slower, lower limits.
#   "qwen/qwen3.8-27b"     Alibaba Qwen (a preview model, which Groq may withdraw at short notice).
#   Groq retires models from time to time. Current list: https://console.groq.com/docs/models
MODEL = "openai/gpt-oss-20b"

print("Settings configured.")

## Step 2: Fetch issues from GitHub (run this cell)

This cell uses the **GitHub REST API**. No AI is involved at this stage, only data. The quick test at the end shows real issues immediately.

In [ ]:
import requests

GITHUB_API = "https://api.github.com"
HEADERS = {"Accept": "application/vnd.github+json"}
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"


def clean_repo_name(text):
    """Accepts 'owner/repo' or a full GitHub URL and returns 'owner/repo'."""
    text = text.strip().removeprefix("https://").removeprefix("http://").removeprefix("github.com/").strip("/")
    parts = text.split("/")
    if len(parts) < 2 or not parts[0] or not parts[1]:
        raise ValueError("Enter a repository in the form owner/name, for example ollama/ollama.")
    return f"{parts[0]}/{parts[1]}"


def github_get(path, params=None):
    response = requests.get(f"{GITHUB_API}{path}", headers=HEADERS, params=params, timeout=20)
    if response.status_code == 404:
        raise ValueError("Repository not found. Check the spelling (format: owner/name).")
    if response.status_code in (403, 429):
        raise ValueError("GitHub rate limit reached. Wait a few minutes, or add a GITHUB_TOKEN secret (see the top of the notebook).")
    response.raise_for_status()
    return response.json()


def fetch_beginner_issues(repo):
    """Returns (issues, description_of_how_they_were_found)."""
    repo = clean_repo_name(repo)

    # 1. Find the repository's labels that indicate beginner-friendly issues.
    labels = github_get(f"/repos/{repo}/labels", {"per_page": 100})
    beginner_labels = [l["name"] for l in labels if any(w in l["name"].lower() for w in BEGINNER_LABEL_WORDS)]

    # 2. Fetch open issues with those labels.
    issues = {}
    for label in beginner_labels:
        for item in github_get(f"/repos/{repo}/issues", {"labels": label, "state": "open", "per_page": MAX_ISSUES}):
            if "pull_request" not in item:  # GitHub also lists pull requests as issues; these are excluded
                issues[item["number"]] = item
    found_how = f"labelled {', '.join(repr(l) for l in beginner_labels)}"

    # 3. If there are no beginner-labelled issues, use the most recent open issues instead.
    if not issues:
        for item in github_get(f"/repos/{repo}/issues", {"state": "open", "per_page": 30}):
            if "pull_request" not in item:
                issues[item["number"]] = item
        found_how = "recent open issues (this repository currently has no open beginner-labelled issues)"

    # STUDENT TODO (Level 3): sort or filter the issues here.
    #   Example: keep only issues with fewer than 5 comments: [i for i in ... if i["comments"] < 5]
    simple = [{
        "number": i["number"],
        "title": i["title"],
        "url": i["html_url"],
        "labels": [l["name"] for l in i["labels"]],
        "comments": i["comments"],
        "body": (i.get("body") or "")[:800],  # only the beginning, to keep processing fast
    } for i in issues.values()]
    return simple[:MAX_ISSUES], found_how


# Quick test: fetch real issues from the default repository.
try:
    test_issues, how = fetch_beginner_issues(DEFAULT_REPO)
    print(f"Found {len(test_issues)} issues in {DEFAULT_REPO} ({how}):\n")
    for issue in test_issues:
        print(f"  #{issue['number']}: {issue['title']}")
except Exception as e:
    print("Error:", e)

## Step 3: Explain the issues using AI (run this cell)

All issues are sent to the AI model in a **single** request (faster, and uses less of the free limit), and the response is requested in JSON format.

In [ ]:
import json

DIFFICULTY_LABELS = {"Easy": "Difficulty: Easy", "Medium": "Difficulty: Medium", "Hard": "Difficulty: Hard"}  # STUDENT TODO (Level 2)


def explain_issues(issues):
    """Returns {issue_number: explanation}. Returns an empty result in demonstration mode or if the response is invalid."""
    issue_text = "\n\n---\n\n".join(
        f"Issue #{i['number']}: {i['title']}\nLabels: {', '.join(i['labels']) or 'none'}\n{i['body']}" for i in issues
    )
    messages = [
        {"role": "system", "content": EXPLAIN_INSTRUCTIONS},
        {"role": "user", "content": issue_text},
    ]
    reply = ask_llm(messages, temperature=TEMPERATURE, json_mode=True, demo_reply="{}")
    if reply.startswith("Error:"):
        raise ValueError(reply)
    try:
        explained = json.loads(reply).get("issues", [])
        return {int(e["number"]): e for e in explained if "number" in e}
    except (json.JSONDecodeError, ValueError, TypeError):
        return {}


def make_cards(issues, explanations):
    """Combines issues and AI explanations into Markdown cards."""
    cards = []
    for i in issues:
        e = explanations.get(i["number"])
        card = f"### [#{i['number']}: {i['title']}]({i['url']})\n"
        card += f"Labels: {', '.join(i['labels']) or 'none'} | Comments: {i['comments']}\n\n"
        if e:
            card += f"**{DIFFICULTY_LABELS.get(e.get('difficulty'), 'Difficulty: ' + str(e.get('difficulty', 'unknown')))}**\n\n"
            card += f"**Summary:** {e.get('summary', '')}\n\n"
            card += f"**Skills required:** {', '.join(e.get('skills', []))}\n\n"
            card += f"**Suggested first step:** {e.get('first_step', '')}\n"
        elif DEMO_MODE:
            card += "*Demonstration mode: add a GROQ_API_KEY to generate a plain-language explanation.*\n"
        else:
            card += "*No explanation was generated for this issue. Please try again.*\n"
        # STUDENT TODO (Level 3): add further information to each card, such as a draft comment requesting assignment.
        cards.append(card)
    return "\n\n---\n\n".join(cards)


# Quick test: explain the issues fetched above.
try:
    print(make_cards(test_issues, explain_issues(test_issues)))
except Exception as e:
    print("Error:", e)

## Step 4: Launch the application

Run this cell to launch the application and create a **public link**. Re-run this cell after changing any cell above.

In [ ]:
import gradio as gr

gr.close_all()  # closes any previous version of the application that is still running


def on_search(repo):
    try:
        issues, how = fetch_beginner_issues(repo)
        if not issues:
            return f"No open issues were found in **{repo}**. Try another repository."
        explanations = explain_issues(issues)
    except Exception as e:
        raise gr.Error(str(e))
    return f"## {len(issues)} issues found in `{clean_repo_name(repo)}`\n*Source: {how}*\n\n---\n\n" + make_cards(issues, explanations)


with gr.Blocks(theme=gr.themes.Soft(), title=APP_TITLE) as demo:  # STUDENT TODO (Level 2): alternative theme: gr.themes.Glass()
    gr.Markdown(f"# {APP_TITLE}\n{APP_DESCRIPTION}")
    with gr.Row():
        repo_box = gr.Textbox(value=DEFAULT_REPO, label="GitHub repository (owner/name or URL)", scale=4)
        search_btn = gr.Button("Find issues", variant="primary", scale=1)
    # STUDENT TODO (Level 1): add repositories of interest as examples.
    gr.Examples(["ollama/ollama", "huggingface/transformers", "gradio-app/gradio", "freeCodeCamp/freeCodeCamp"], inputs=repo_box)
    results_md = gr.Markdown()

    search_btn.click(on_search, inputs=repo_box, outputs=results_md)
    repo_box.submit(on_search, inputs=repo_box, outputs=results_md)

demo.launch(share=True)

## Level 3: Optional extensions

**1. Real contribution (recommended for the demonstration).** Use the tool to find an issue the team could realistically work on. Read it in full and present a plan in the demonstration. If time allows, comment on the issue to offer help.

**2. Skill filter.** Add a menu (for example: Any, Documentation, Python, JavaScript, Design) and instruct the AI to recommend only matching issues, or filter by label.

**3. Suitability ranking.** Ask the AI to include a beginner suitability score from 1 to 10 for each issue in the JSON, and sort the cards by that score.

**4. Draft comment.** Add a `"draft_comment"` field to the JSON, containing a courteous message requesting to be assigned to the issue, and show it on each card.

**5. Repository summary.** Fetch the repository's README (`GET /repos/{owner}/{repo}/readme`, which is base64-encoded) and ask the AI for a three-sentence summary of the project to show above the results.

**6. Repository comparison.** Allow two repositories to be entered and show which has more beginner-friendly issues.

## Demonstration checklist
- [ ] A repository named by the audience, with real issues found and explained live
- [ ] Issue cards showing summary, difficulty, required skills and first step
- [ ] The team can explain what an API is and how the tool combines two (GitHub and Groq)
- [ ] One issue selected by the team, with a justification

## Troubleshooting

| Issue | Resolution |
|---|---|
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| The application does not reflect recent changes | Re-run the edited cell **and** the "Launch the application" cell. |
| The notebook remains in demonstration mode | Add `GROQ_API_KEY` in Secrets, enable **Notebook access**, and re-run the setup cell. |
| "Usage limit reached" error | Wait about one minute. Free keys allow a limited number of requests per minute. |
| `SyntaxError` | Usually caused by a missing quotation mark `"`, bracket `)` or comma. Check the line shown in the error. |
| Unexpected behaviour | Select **Runtime → Restart session**, then run all cells from the top. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |